# Schneider Electric India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.se.com (Phenom People portal — requires Selenium)

**ATS:** Phenom People (custom portal — requests return 403)

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path
SCRIPTS_DIR = Path.home() / 'Job_Scrapers' / 'All_Scripts'
sys.path.insert(0, str(SCRIPTS_DIR))
from scraper_utils import *
from bs4 import BeautifulSoup
from datetime import datetime
LOCATION_FILTER = 'India'
print('Imports loaded. Date:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-04-02 10:29:41


In [3]:
COMPANY = 'Schneider_Electric'
OUTPUT_DIR = get_output_dir(COMPANY)
print(f'Output directory: {OUTPUT_DIR}')

Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Schneider_Electric/Outputs/2026_04_02


In [4]:
print('=' * 60)
print('SCHNEIDER ELECTRIC INDIA JOB SCRAPER')
print('Portal: careers.se.com (Phenom People — Selenium required)')
print('=' * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE_URL = 'https://careers.se.com'
# Phenom People uses facet-based URL params; India filter via location or country
START_URL = f'{BASE_URL}/jobs?location=India&country=India'
MAX_JOBS = 500
JD_FETCH_LIMIT = 50


def fetch_jd_detail(driver, url):
    if not url: return ''
    try:
        driver.get(url); time.sleep(random.uniform(2, 3))
        soup = BeautifulSoup(driver.page_source, 'lxml')
        for sel in ["[class*='job-description']", "[class*='description']",
                    "[class*='jd-content']", 'article', 'main']:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100: return el.get_text(' ', strip=True)
        body = soup.select_one('body')
        return body.get_text(' ', strip=True)[:6000] if body else ''
    except Exception as e:
        print(f'    [WARN] JD fetch failed: {e}'); return ''


se_jobs = []
seen_ids = set()

driver = setup_selenium()
try:
    driver.get(START_URL)
    time.sleep(10)
    try:
        WebDriverWait(driver, 25).until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "[class*='job-card'],[class*='job-result'],[data-ph-at-id*='job'],a[href*='/jobs/']")))
    except: time.sleep(8)

    for page in range(40):
        soup = BeautifulSoup(driver.page_source, 'lxml')
        # Phenom uses data-ph-at-id attributes and specific class patterns
        cards = (soup.select("[data-ph-at-id*='job']") or
                 soup.select("[class*='job-card']") or
                 soup.select("[class*='search-result-item']") or
                 soup.select("[class*='job-result']"))
        if not cards:
            job_links = soup.select("a[href*='/jobs/'], a[href*='/job/']")
            seen = set()
            for link in job_links:
                p = link.find_parent(['li', 'div', 'article'])
                if p and id(p) not in seen: cards.append(p); seen.add(id(p))
        new_count = 0
        for card in cards:
            title_el = (card.select_one("[data-ph-at-id*='title']") or
                        card.select_one("[class*='title'] a") or
                        card.select_one('h2 a') or card.select_one('h3 a') or
                        card.select_one("a[href*='/jobs/']") or card.select_one('a'))
            title = title_el.get_text(strip=True) if title_el else ''
            if not is_valid_job_title(title): continue
            href = title_el.get('href', '') if title_el else ''
            job_url = href if href.startswith('http') else (BASE_URL + href if href else '')
            job_id = href.rstrip('/').split('/')[-1] if href else str(abs(hash(title + job_url)))
            loc_el = (card.select_one("[data-ph-at-id*='location']") or
                      card.select_one("[class*='location']"))
            loc = loc_el.get_text(strip=True) if loc_el else 'India'
            dept_el = card.select_one("[class*='department']") or card.select_one("[class*='category']")
            dept = dept_el.get_text(strip=True) if dept_el else ''
            if job_id not in seen_ids:
                seen_ids.add(job_id)
                se_jobs.append({
                    'job_id': job_id, 'title': title, 'company_name': 'Schneider Electric',
                    'job_url': job_url, 'source_api_url': START_URL,
                    'business_unit': dept, 'raw_jd_text': card.get_text(' ', strip=True),
                    'location_city': loc.split(',')[0].strip(), 'location_country': 'India',
                    'industry': 'Energy Management / Industrial Automation',
                    'date_posted': datetime.now().strftime('%Y-%m-%d'),
                    'is_active': True, 'salary_currency': 'INR', 'source_platform': 'Phenom'
                })
                new_count += 1
        print(f'  Page {page+1}: {new_count} new jobs (total: {len(se_jobs)})')
        if new_count == 0 and page > 0: break
        if len(se_jobs) >= MAX_JOBS: break
        # Phenom pagination: load-more button or infinite scroll
        try:
            load_more = driver.find_element(By.CSS_SELECTOR,
                "button[data-ph-at-id*='load-more'],button[class*='load-more'],"
                "a[aria-label*='Next'],a[aria-label*='next'],[class*='next'] a")
            driver.execute_script('arguments[0].click();', load_more)
            time.sleep(4)
        except:
            # Try infinite scroll
            prev_len = len(se_jobs)
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            time.sleep(3)
            new_soup = BeautifulSoup(driver.page_source, 'lxml')
            new_cards = new_soup.select("[data-ph-at-id*='job'],[class*='job-card'],a[href*='/jobs/']")
            if len(new_cards) <= len(cards) or len(se_jobs) == prev_len: break

    # Fetch full JDs
    limit = min(len(se_jobs), JD_FETCH_LIMIT)
    if limit:
        print(f'\n  Fetching JDs for {limit} jobs...')
        for i, job in enumerate(se_jobs[:limit]):
            if job.get('raw_jd_text') and len(job['raw_jd_text']) > 200: continue
            jd = fetch_jd_detail(driver, job['job_url'])
            if jd: se_jobs[i]['raw_jd_text'] = jd
            if (i + 1) % 10 == 0: print(f'    Fetched {i+1}/{limit} JDs')
except Exception as e:
    print(f'  [ERROR] {e}'); import traceback; traceback.print_exc()
finally:
    driver.quit()

print(f'\nTotal Schneider Electric India jobs: {len(se_jobs)}')

SCHNEIDER ELECTRIC INDIA JOB SCRAPER
Portal: careers.se.com (Phenom People — Selenium required)


  Page 1: 0 new jobs (total: 0)



Total Schneider Electric India jobs: 0


In [5]:
df_se = save_results(se_jobs, 'Schneider_Electric', OUTPUT_DIR)
if df_se is not None:
    cols = ['title','location_city','seniority_level','business_unit','job_url']
    cols = [c for c in cols if c in df_se.columns]
    print(df_se[cols].head(10).to_string())

  [WARN] No jobs found for Schneider_Electric
